# Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive


# Lab: Deep-Dive into Google Colab


|    Group Members      | Student Id |
|:----------------------|:-----------:|
| Dinal Maha Vidanelage | 2425591 |
| Olga Chitembo         | 2412122 |
| Sandip Ranjit         | 2424444    |
| Twe He Gam            | 2412112|

*Note: To prevent code overwrites and merge conflicts, one member created the primary master notebook and everyone worked on a cloned copy to test edge cases (such as GPU memory limits and precision checks).*

## Overview of Google Colab and Cell Types

Google Colab is an interactive, browser-based cloud environment based on Jupyter Notebooks. It allows executable code, data visualizations, and structured text to coexist in a single document while providing access to cloud-hosted CPU and GPU runtimes.

---

#### Primary Cell Types

1. **Code Cells**
   * **Function:** Executable cells used to run code, load datasets, and perform matrix calculations.
   * **Behavior:** Code is sent directly to the active **Python Kernel** running on the cloud virtual machine. Any variables, functions, or imports created in a code cell remain saved in memory (RAM/VRAM) and can be accessed by subsequent cells. Outputs—such as printed text, terminal logs, or plotted charts—appear directly beneath the cell.

2. **Text (Markdown) Cells**
   * **Function:** Static cells used for documenting methodology, providing theoretical context, and structuring the report.
   * **Behavior:** These cells do not execute Python code. Instead, they render standard **Markdown** (for headings, bulleted lists, and tables) alongside **LaTeX** for formatting mathematical models (e.g., $C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}$).

---

#### Execution Modes Within Code Cells

* **Standard Python:** Normal Python statements (e.g., `import torch`, `c = torch.matmul(a, b)`) execute within the persistent Python process.
* **Shell Commands (`!`):** Prefixing a command with an exclamation point (e.g., `!nvidia-smi` or `!ls`) executes instructions in a temporary system terminal shell outside the Python environment.
* **Magic Commands (`%` or `%%`):** Instructions starting with `%` (line magic) or `%%` (cell magic) control the notebook environment directly, such as changing working directories (`%cd`) or timing cell execution (`%timeit`).

In [ ]:
x = 10
print(x)

10


In [ ]:
!pwd

/content


In [ ]:
!ls

drive  sample_data


In [ ]:
%cd /content/drive/MyDrive/Data Handling and Machine Learning

/content/drive/.shortcut-targets-by-id/1wIXBoTAvdbYOXIGoJehJOd8zm1gA7wE2/Data Handling and Machine Learning


In [ ]:
import os

print("Python version:")
print(os.sys.version)

print("\nCurrent working directory:")
print(os.getcwd())

Python version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Current working directory:
/content/drive/.shortcut-targets-by-id/1wIXBoTAvdbYOXIGoJehJOd8zm1gA7wE2/Data Handling and Machine Learning


# Task 1A - Working Directories: The Classic !cd vs %cd Experiment

**Prediction**

What do you expect this to print?
```
!cd /tmp
```
followed by, in a new cell:
```
!pwd
```

- !pwd will still print out /content becuse !cd /tmp was working in its own shell

In [ ]:
!cd /tmp

In [ ]:
!pwd

/content/drive/.shortcut-targets-by-id/1wIXBoTAvdbYOXIGoJehJOd8zm1gA7wE2/Data Handling and Machine Learning


In [ ]:
%cd /tmp

/tmp


In [ ]:
!pwd

/tmp


In [ ]:
import os
print(os.getcwd())

/tmp


# Task 1B - Important Counterexample: ! Can Change the Filesystem

In [ ]:
!mkdir -p my_project
!touch my_project/data.txt

In [ ]:
!ls -l my_project

total 0
-rw-r--r-- 1 root root 0 Aug 21 13:25 data.txt


# Task 1C - Shell Variables vs Python Variables

**Experiment 1 - Shell variable**

In [ ]:
!MY_VARIABLE="Colab" && echo "Inside this command: $MY_VARIABLE"

Inside this command: Colab


In [ ]:
!echo "In a new shell: $MY_VARIABLE"

In a new shell: 


**Experiment 2 - Python variable**

In [ ]:
my_variable = "Colab"
print(my_variable)

Colab


In [ ]:
print(my_variable)

Colab


**Experiment 3 - Python environment variables**

In [ ]:
import os

os.environ["COURSE_NAME"] = "Cloud Computing"
print(os.environ["COURSE_NAME"])

Cloud Computing


In [ ]:
!echo $COURSE_NAME

Cloud Computing


# Task 1D - A Separate Python Process

In [ ]:
shared_data = [10, 20, 30]
print("Notebook kernel: ", shared_data)

Notebook kernel:  [10, 20, 30]


In [ ]:
!python3 -c "print('External Python process started')"

External Python process started


In [ ]:
!python3 -c "print('shared_data' in globals())"

False


# Task 1E - `%run` vs `!python`

In [ ]:
%%writefile experiment.py

experiment_value = 42
print("Script executed")


Writing experiment.py


In [ ]:
%run experiment.py

Script executed


In [ ]:
print(experiment_value)

42


In [ ]:
!python3 experiment.py

Script executed


# Task 1F - Summary
Complete the following table in your notebook.

| Operation                   | Persists across cells? |Explanation                                |
| --------------------------- | :---------------------: | ------------------------------------------ |
| `!cd /tmp` | **False** | Changes only the shell's working directory |
| `%cd /tmp` | **True** | Changes the IPython working directory permanently |
| `!MY_VAR=1` | **False** | Shell-local variable that vanishes when the subshell terminates |
| `my_var = 1` | **True** | Python variable in kernel memory |
| `!mkdir test` | **True*** | Filesystem modification |
| `!touch file.txt` | **True*** | Filesystem modification |
| `!export KEY=value` | **False** | Export applies to that shell process |
| `os.environ["KEY"]="value"` | **True** | Changes the Python process environment |
| `!python3 script.py` | **False** | Runs in a separate process; variables inside script are lost |
| `%run script.py` | **True** | Runs inside IPython; variables/functions load into kernel memory |

*\*Note: Filesystem changes persist until the runtime is reset, disconnected, or manually deleted.*

# Task 2 - Working with Files and Python Code

**2.1 Creating a Python file with `%%writefile**`

In [ ]:
%%writefile math_utils.py

def square_sum(n):
  return sum(i * i for i in range(n))

def factorial(n):
  result = 1
  for i in range(1, n + 1):
    result *= i
  return result


Writing math_utils.py


- Verify that the file exists:

In [ ]:
!ls -l math_utils.py

-rw-r--r-- 1 root root 150 Aug 21 13:26 math_utils.py


- Inspect the file's contents

In [ ]:
!cat math_utils.py


def square_sum(n):
  return sum(i * i for i in range(n))

def factorial(n):
  result = 1
  for i in range(1, n + 1):
    result *= i
  return result


**2.2 Execute the file with `%run`**

In [ ]:
%run math_utils.py

In [ ]:
print(square_sum(10))
print(factorial(5))

285
120


**2.3 Compare `%run` with ordinary Python import**

In [ ]:
import math_utils

print(math_utils.square_sum(10))

285


| Method |	Purpose |
| ------ | ------- |
|`%run file.py`	| Execute a script through IPython |
|`import module`	| Import a Python module |
|`!python file.py`	| Start a separate Python process and execute the script |


# Task 3 - Measuring Python Performance

3.1 Why timing matters

In [ ]:
%timeit sum(i * i for i in range(1_000_000))

90.4 ms ± 7.97 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
import numpy as np

%timeit np.sum(np.arange(1_000_000, dtype=np.int64) ** 2)

3.99 ms ± 71.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


`NumPy` implementation is much faster because of following reasons:
- **Compiled C Code:** Runs pre-compiled machine code directly, skipping Python's slow step-by-step interpreter.

- **Vectorization:** Computes millions of operations at once using specialized CPU instructions instead of item-by-item for loops.

- **Contiguous Memory:** Stores numbers right next to each other in RAM, making data retrieval virtually instant for the CPU cache.

- **No Object Overhead:** Operates on raw fixed-type numbers rather than heavy, dynamically typed Python integer objects.




# Task 4 - Inspecting the Cloud Hardware

## 4.1 Inspect the CPU

In [ ]:
!lscpu | grep -E "Model name|Socket|Thread|NUMA|CPU\(s\)"

CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0,1


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       906Mi       8.5Gi       2.0Mi       3.3Gi        11Gi
Swap:             0B          0B          0B


## 4.2 Inspect the operating system

In [ ]:
!cat /etc/os-release

PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


# Task 5 - GPU Acceleration

## 5.2 Inspect the GPU

In [ ]:
!nvidia-smi

Fri Aug 21 13:28:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

- **GPU Model:** Tesla T4

- **Total GPU Memory:** 15,360 MiB (~15 GB)

- **CUDA Version:** 13.0

- **Current GPU Memory Usage:** 0 MiB / 15360 MiB (0% utilized)

## 5.3 Check GPU availability from PyTorch

In [ ]:
import torch

print("PyTorch version: ", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU: ", torch.cuda.get_device_name(0))
  print(
     "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
  )

PyTorch version:  2.11.0+cu128
CUDA available:  True
GPU:  Tesla T4
GPU memory: 14.56 GB


# Task 6 - CPU vs GPU Matrix Multiplication

## 6.2 Benchmark implementation

In [ ]:
import torch
import time

def benchmark_matmul(n=4000):

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Enable a GPU runtime first.")

    print(f"Matrix size: {n} x {n}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    # -------------------------
    # CPU
    # -------------------------
    a_cpu = torch.randn(n, n, dtype=torch.float32)
    b_cpu = torch.randn(n, n, dtype=torch.float32)

    # Warm up CPU execution
    _ = torch.matmul(a_cpu, b_cpu)

    start = time.perf_counter()
    c_cpu = torch.matmul(a_cpu, b_cpu)
    cpu_time = time.perf_counter() - start

    # -------------------------
    # GPU
    # -------------------------
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    # Warm up GPU
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    start = time.perf_counter()

    c_gpu = torch.matmul(a_gpu, b_gpu)

    # GPU operations are asynchronous.
    # Wait for the GPU before stopping the timer.
    torch.cuda.synchronize()

    gpu_time = time.perf_counter() - start

    speedup = cpu_time / gpu_time

    print(f"CPU time: {cpu_time:.4f} s")
    print(f"GPU time: {gpu_time:.4f} s")
    print(f"GPU speedup: {speedup:.2f}x")

    return cpu_time, gpu_time, speedup

In [ ]:
cpu_time, gpu_time, speedup = benchmark_matmul(4000)

Matrix size: 4000 x 4000
GPU: Tesla T4
CPU time: 1.0139 s
GPU time: 0.0457 s
GPU speedup: 22.18x


# Task 7 - Understanding CUDA Asynchrony

## 7.1 Deliberately perform an incorrect benchmark

**Writing the benchmark function as reusable Python module**

In [ ]:
%%writefile benchmark_v2.py

import torch
import time

def benchmark_matmul(n=4000, sync=True):

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Enable a GPU runtime first.")

    print(f"Matrix size: {n} x {n}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    # -------------------------
    # CPU
    # -------------------------
    a_cpu = torch.randn(n, n, dtype=torch.float32)
    b_cpu = torch.randn(n, n, dtype=torch.float32)

    # Warm up CPU execution
    _ = torch.matmul(a_cpu, b_cpu)

    start_cpu = time.perf_counter()
    c_cpu = torch.matmul(a_cpu, b_cpu)
    cpu_time = time.perf_counter() - start_cpu

    # -------------------------
    # GPU
    # -------------------------
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    # Warm up GPU
    _ = torch.matmul(a_gpu, b_gpu)
    if sync:
        torch.cuda.synchronize()

    start_gpu = time.perf_counter()
    c_gpu = torch.matmul(a_gpu, b_gpu)

    if sync:
        torch.cuda.synchronize() # Wait for GPU to finish

    gpu_time = time.perf_counter() - start_gpu

    speedup = cpu_time / gpu_time if gpu_time > 0 else 0.0

    return cpu_time, gpu_time, speedup

Writing benchmark_v2.py


In [ ]:
from benchmark_v2 import benchmark_matmul

_, incorrect_time, _ = benchmark_matmul(n=4000, sync=False)

print(f"Without synchronization: {incorrect_time:.7f} s")

Matrix size: 4000 x 4000
GPU: Tesla T4
Without synchronization: 0.0001063 s


In [ ]:
from benchmark_v2 import benchmark_matmul

_, correct_time, _ = benchmark_matmul(n=4000, sync=True)

print(f"With synchronization: {correct_time:.7f} s")

Matrix size: 4000 x 4000
GPU: Tesla T4
With synchronization: 0.0445644 s


# Task 8 - Investigate the Effect of Problem Size

In [ ]:
from benchmark_v2 import benchmark_matmul

for size in [500, 1000, 2000, 3000]:
    print("\n" + "=" * 50)
    cpu_t, gpu_t, speedup = benchmark_matmul(size, sync=True)
    print(f"Summary ({size}x{size}): CPU = {cpu_t:.6f}s | GPU = {gpu_t:.6f}s | Speedup = {speedup:.2f}x")


Matrix size: 500 x 500
GPU: Tesla T4
Summary (500x500): CPU = 0.002447s | GPU = 0.000183s | Speedup = 13.35x

Matrix size: 1000 x 1000
GPU: Tesla T4
Summary (1000x1000): CPU = 0.017582s | GPU = 0.000871s | Speedup = 20.18x

Matrix size: 2000 x 2000
GPU: Tesla T4
Summary (2000x2000): CPU = 0.131033s | GPU = 0.005955s | Speedup = 22.00x

Matrix size: 3000 x 3000
GPU: Tesla T4
Summary (3000x3000): CPU = 0.420794s | GPU = 0.019299s | Speedup = 21.80x


**Task 10.3: Benchmark Table of Olga's device**

| Matrix size	|   CPU time	|   GPU time	|    GPU speedup |
| ----------- | :---------: | :---------: | :------------: |
| 500 × 500		| 0.002447s   | 0.000183s   | 13.35x         |    
| 1000 × 1000	| 0.017582s   | 0.000871s   | 20.18x         |
| 2000 × 2000	| 0.131033s   | 0.005955s   | 22.00x         |
| 3000 × 3000	| 0.420794s   | 0.019299s   | 21.80x         |

1. Does GPU speedup increase with problem size?
- Yes. GPU speedup tends to become more noticeable as the problem becomes larger. Larger matrix operations provide more parallel work for the GPU and can better amortize fixed overhead.

2. Is the GPU always faster?
- No. Small workloads may run faster on a CPU because the GPU has additional launch and setup overhead.

3. Why might a small matrix not benefit as much?
- The actual computation may be too small to fully utilize the GPU.

4. What role might data transfer and kernel-launch overhead play?
- Kernel Launch Overhead: Sending a execution command from the CPU to the GPU takes a fixed amount of setup time (a few microseconds). For tiny inputs, launching the job takes longer than calculating the result.

- PCIe Bus Transfer: Moving data between CPU and GPU memory takes time. For small computations, these costs can represent a large portion of the total execution time. For larger workloads, the actual parallel computation becomes much more significant compared with these fixed costs.

# Task 9 - Verify CPU and GPU Results

In [ ]:
import torch

n = 4000

# 1. Create tensors on CPU
a_cpu = torch.randn(n, n, dtype=torch.float32)
b_cpu = torch.randn(n, n, dtype=torch.float32)

# 2. Multiply on CPU
c_cpu = torch.matmul(a_cpu, b_cpu)

# 3. Move inputs to GPU and multiply
a_gpu = a_cpu.to("cuda")
b_gpu = b_cpu.to("cuda")
c_gpu = torch.matmul(a_gpu, b_gpu)

# 4. Compare CPU and GPU results
difference = torch.max(torch.abs(c_cpu - c_gpu.cpu()))

print("Maximum absolute difference:", difference.item())

Maximum absolute difference: 0.001007080078125


# Task 10 - Document the Experiment with Markdown and $\LaTeX$

## 10.2 Mathematical model
Include:

$$
C_{ij} = \sum_{k=1}^{n} A_{ik}B_{kj}
$$

Explain that standard matrix multiplication requires $\mathcal{O}(n^3)$ arithmetic work.

---

**Answer:**  For two matrices A and B, matrix multiplication is defined as:

$$
C_{ij} = \sum_{k=1}^{n} A_{ik}B_{kj}
$$

For two $n \times n$ matrices, the standard matrix multiplication
 requires approximately $\mathcal{O}(n^3)$ arithmetic operations.

## 10.4 Explain the speedup

The GPU can accelerate matrix multiplication because the calculation contains a large number of independent operations that can be executed in parallel. GPUs contain many processing units and are designed for highly parallel workloads.
CUDA operations can execute asynchronously, meaning that the CPU may continue executing Python code before a GPU operation has finished. Therefore, CPU-side timing can underestimate the actual GPU computation time if synchronization is not performed.
`torch.cuda.synchronize()` forces the CPU to wait until the GPU has completed its queued operations. This makes the measured interval include the actual GPU computation.
GPU acceleration is generally more useful for larger workloads because larger calculations provide enough parallel work to offset overhead such as GPU kernel launches and data transfers. For small workloads, these overheads can make a GPU less advantageous than a CPU.

>**Important:** GPU benchmarks must account for asynchronous execution. CPU-side timing without synchronization can underestimate the actual GPU computation time.


# Task 11 - Persistent Storage with Google Drive

## 11.2 Inspect Drive

In [ ]:
!ls -lah "/content/drive/MyDrive" | head

total 8.5K
drwx------ 3 root root 4.0K Aug 19 19:08 Colab Notebooks
-rw------- 1 root root   16 Aug 19 19:11 colab_text.txt
drwx------ 4 root root 4.0K Aug 19 19:08 Data Handling and Machine Learning


## 11.3 Compare temporary and persistent storage

In [ ]:
!echo "temporary data" > /content/temporary.txt

In [ ]:
!echo "persistent data" > "/content/drive/MyDrive/colab_text.txt"

In [ ]:
!cat /content/temporary.txt
!cat /content/drive/MyDrive/colab_text.txt

temporary data
persistent data



`/content/` is storage belonging to the temporary colab runtime. files stored there are generally lost when runtime is terminated or reset.

`/content/drive/MyDrive/` refers to the google drive storage mounted into the colab runtime. files stored there persist independently of runtime.

# Final Reflection

### Question 1
What is the difference between:

`print(...)`: Python code executed by the python kernel used to display values or text.

`!command`: tells Ipython to execute a linux shell command in a seperate sub-process.

`%magic`: invokes an Ipython magic command handled by Ipython itself. It provides functionality for interactive notebook environments.

---

### Question 2

Why `!cd /tmp` vs `%cd /tmp` behaves differently?

- `!cd /tmp` !cd runs inside a temporary shell sub-process. when the shell exits, the change disapears.

- `%cd /tmp` This changes the working directory of the working Ipython process so future shell commands inherit that directory.

---

### Question 3

Why `!touch file.txt` creates a persistent file?

touch modifies the system by creating a file. the file is stored in the filesystem not the shell's memory so the file remains available even when the shell is terminated.

---

### Question 4

What is the difference between the Python kernel and a separate process started with `!python3 script.py`?

- **Python Kernel:** the persistent python process associated with the notebook. variables created in the kernel remain available across cells

- `!python3 script.py`: starts a seperate python process which has its own memory. variables created in that process are not automatically available in the kernel.

---

### Question 5

Why is `torch.cuda.synchronize()` important when measuring GPU execution time?

GPU operations can operate asynchronously. The CPU can submit a GPU operation and continue running without waiting for the GPU to finish which can produce inacurate timing. `torch.cuda.sychronize()` forces the CPU to wait until the GPU finishes its pending operations.

---

### Question 6

Why is a GPU not automatically faster than a CPU for every computation?

GPUs are designed for highly parallel workloads. for a small computation , overheads can be larger than the time saved by GPU parallelism.

---

### Question 7

Why should important experiment data not be stored only in `/content/`?

The `/content/` directory belongs to the temporary colab vm, when runtime is disconnected the files stored only in the vm may be deleted.